# Deep Hedging — Phase 1, brique 1 : simuler des trajectoires

Objectif : générer proprement des trajectoires de prix sous un brownien géométrique, et **vérifier** qu'elles collent à la théorie (S_t log-normal, E[S_t] = S0·exp(mu·t), log-rendement gaussien).

Aucun réseau, aucune couverture encore. On veut juste un moteur de simulation dont on est sûr, parce que tout le reste s'appuiera dessus.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(0)   # générateur à graine fixe, pour la reproductibilité

## La fonction de simulation

Modèle : `dS = mu·S·dt + sigma·S·dW`, dont la solution **exacte** est

    S_{t+dt} = S_t · exp( (mu - sigma²/2)·dt + sigma·sqrt(dt)·Z ),   Z ~ N(0,1).

On utilise cette solution exacte, pas un schéma d'Euler approché : on connaît la forme fermée du GBM, donc autant l'utiliser. C'est exact quel que soit `dt`.

In [ ]:
def simulate_gbm(S0, mu, sigma, T, n_steps, n_paths, rng=RNG):
    """Simule n_paths trajectoires de brownien géométrique sur [0, T].
    Renvoie S de forme (n_paths, n_steps + 1) : colonnes = dates t_0..t_n."""
    dt = T / n_steps                                   # pas de temps entre deux dates

    # Un incrément brownien standard normal par (trajectoire, pas de temps).
    Z = rng.standard_normal((n_paths, n_steps))

    # Incréments du LOG du prix : d(ln S) = (mu - sigma^2/2) dt + sigma dW,
    # avec dW = sqrt(dt) Z. Le -sigma^2/2 est la correction d'Ito.
    log_increments = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z

    # log-prix en t_k = ln S0 + somme des incréments jusqu'à k. On cumule sur le
    # temps (axis=1) et on colle une colonne de zéros pour t_0.
    zeros = np.zeros((n_paths, 1))
    log_paths = np.concatenate([zeros, np.cumsum(log_increments, axis=1)], axis=1)

    # retour du log au prix : S = S0 exp(log_paths)
    S = S0 * np.exp(log_paths)
    return S

## Simulation et vérification contre la théorie

In [ ]:
# Paramètres du marché (un an, 252 jours de bourse, vol 20 %).
S0, mu, sigma, T = 100.0, 0.05, 0.20, 1.0
n_steps, n_paths = 252, 200_000

S = simulate_gbm(S0, mu, sigma, T, n_steps, n_paths)
ST = S[:, -1]                        # prix final de chaque trajectoire

# Vérif 1 : E[S_T] = S0 exp(mu T)  (le fait qu'on a démontré via la martingale)
print(f"E[S_T] empirique = {ST.mean():.4f}   |   théorie S0*exp(mu*T) = {S0*np.exp(mu*T):.4f}")

# Vérif 2 : ln(S_T/S0) ~ Normal( (mu - sigma^2/2) T , sigma^2 T )
logret = np.log(ST / S0)
print(f"moyenne log-ret  = {logret.mean():.4f}   |   théorie (mu-sigma^2/2)*T = {(mu-0.5*sigma**2)*T:.4f}")
print(f"variance log-ret = {logret.var():.4f}   |   théorie sigma^2*T        = {sigma**2*T:.4f}")

## Visualisation : trajectoires et distribution terminale

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# 20 trajectoires pour l'oeil
for i in range(20):
    ax1.plot(np.linspace(0, T, n_steps + 1), S[i], lw=0.8)
ax1.set_title("20 trajectoires de brownien géométrique")
ax1.set_xlabel("temps (années)"); ax1.set_ylabel("prix S_t")

# histogramme de S_T + densité log-normale théorique
ax2.hist(ST, bins=120, density=True, alpha=0.6, color="steelblue")
x = np.linspace(ST.min(), np.percentile(ST, 99.5), 400)
m = np.log(S0) + (mu - 0.5 * sigma**2) * T          # moyenne du log
s = sigma * np.sqrt(T)                               # écart-type du log
pdf = np.exp(-(np.log(x) - m)**2 / (2 * s**2)) / (x * s * np.sqrt(2 * np.pi))
ax2.plot(x, pdf, "r", lw=2, label="densité log-normale théorique")
ax2.set_title("Distribution de S_T"); ax2.set_xlabel("S_T"); ax2.legend()

plt.tight_layout()
plt.show()